# <font color="#418FDE" size="6.5" uppercase>**Adversarial Attacks and Security in AI**</font>

>Last update: 20250209.
    
By the end of this Lecture, you will be able to:
- Describe different types of adversarial attacks on AI models.
- Assess vulnerabilities in models and datasets.
- Propose security strategies to mitigate these threats.


## **1. Understanding Adversarial Attack Types: Poisoning, Evasion, and Model Extraction**

### **1.1. Case Studies of Data Poisoning in AI Systems**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20A/image_01_01.png" width="250">



>* Adversaries manipulate AI training data intentionally.
* Spam filters misclassify spam as legitimate emails.

>* Poisoned data alters object detection accuracy.
* Robust validation needed for safety-critical applications.

>* Data poisoning skews AI diagnostic predictions.
* Implement data integrity checks and differential privacy.



### **1.2. Techniques and Impacts of Evasion in Image Recognition**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20A/image_01_02.png" width="250">



>* Evasion attacks exploit AI model vulnerabilities.
* Robust training needed against adversarial manipulations.

>* Diverse techniques generate adversarial examples.
* Attacks undermine AI trust and security.

>* Evasion attacks degrade AI system reliability.
* Robust defenses needed against evasion attacks.



In [ ]:
#@title Python Code - Techniques and Impacts of Evasion in Image Recognition

# This code demonstrates an evasion attack on a simple image recognition model using the Fast Gradient Sign Method (FGSM).
# The purpose of this code is to illustrate how adversarial examples can be generated to fool a neural network model.
# We will use the CIFAR-10 dataset, which consists of 60,000 32x32 color images in 10 classes.
# The model used here is a simple Convolutional Neural Network (CNN) built with TensorFlow/Keras.
# The FGSM attack will be applied to a test image to create an adversarial example.

import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
import numpy as np
import matplotlib.pyplot as plt

# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalize the data
x_train, x_test = x_train / 255.0, x_test / 255.0

# Define a simple CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Train the model
model.fit(x_train, y_train, epochs=3, validation_data=(x_test, y_test))

# Select a test image
image_index = 0
image = x_test[image_index]
image_label = y_test[image_index]

# Function to create adversarial example using FGSM
def create_adversarial_pattern(input_image, input_label):
    # Convert image and label to tensors
    input_image = tf.convert_to_tensor(input_image.reshape((1, 32, 32, 3)))
    input_label = tf.convert_to_tensor(input_label)
    # Use GradientTape to record the operations for automatic differentiation
    with tf.GradientTape() as tape:
        tape.watch(input_image)
        prediction = model(input_image)
        loss = tf.keras.losses.sparse_categorical_crossentropy(input_label, prediction)
    # Get the gradients of the loss w.r.t to the input image
    gradient = tape.gradient(loss, input_image)
    # Get the sign of the gradients to create the perturbation
    signed_grad = tf.sign(gradient)
    return signed_grad

# Generate adversarial example
perturbations = create_adversarial_pattern(image, image_label)
adv_image = image + 0.05 * perturbations.numpy().squeeze()
adv_image = tf.clip_by_value(adv_image, 0, 1)

# Plot original and adversarial images
plt.figure(figsize=(6, 3))
plt.subplot(1, 2, 1)
plt.title('Original Image')
plt.imshow(image)
plt.subplot(1, 2, 2)
plt.title('Adversarial Image')
plt.imshow(adv_image)
plt.show()

# Predict the class of the adversarial image
adv_image = adv_image.numpy().reshape((1, 32, 32, 3))
adv_prediction = model.predict(adv_image)
print(f"Original label: {image_label}, Adversarial prediction: {np.argmax(adv_prediction)}")


### **1.3. Safeguarding Intellectual Property: Challenges and Solutions**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20A/image_01_03.png" width="250">



>* Model extraction attacks infringe on IP rights.
* Implement security measures to protect models.

>* Data poisoning attacks corrupt AI model outputs.
* Implement data validation and anomaly detection systems.

>* Legal frameworks deter IP theft in AI.
* Collaborative approaches enhance IP protection strategies.



### **1.4. Emerging Trends in Adversarial Techniques**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20A/image_01_04.png" width="250">



>* GANs create deceptive inputs for AI systems.
* AI must adapt to evolving adversarial attacks.

>* Transferability enables attacks across different models.
* Cross-model defenses crucial for AI robustness.

>* Adversaries manipulate RL learning processes.
* Need robust RL frameworks against manipulations.



In [ ]:
#@title Python Code - Emerging Trends in Adversarial Techniques

# This code demonstrates a simple adversarial attack using the Fast Gradient Sign Method (FGSM) on a pre-trained
# Convolutional Neural Network (CNN) model using the CIFAR-10 dataset. The purpose is to illustrate how adversarial
# examples can be generated to fool a model, highlighting the emerging trend of adversarial techniques in AI.

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------
# 1. Setup and Data Preparation
# -------------------------------
# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Pretrained ResNet18 was trained on ImageNet, so we use ImageNet stats
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

# Transform: resize, center-crop to 224x224, convert to tensor and normalize
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

# Load CIFAR-10 test set (only testing for demonstration) with batch_size=1 for easier per-sample handling
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=1, shuffle=True)

# Load a pretrained ResNet18 model and set it to evaluation mode
model = models.resnet18(pretrained=True)
model = model.to(device)
model.eval()

# Loss function
criterion = nn.CrossEntropyLoss()

# -------------------------------
# 2. Helper Functions
# -------------------------------
def unnormalize(img):
    """
    Unnormalizes a tensor image using ImageNet statistics.
    The input 'img' is expected to be a tensor of shape (C, H, W).
    """
    # Clone to avoid modifying the original tensor
    img = img.clone()
    for t, m, s in zip(img, imagenet_mean, imagenet_std):
        t.mul_(s).add_(m)
    return img

def fgsm_attack(image, epsilon, data_grad):
    """
    Creates an adversarial example by perturbing the input image using the Fast Gradient Sign Method (FGSM).
    Args:
        image (Tensor): The normalized input image.
        epsilon (float): Magnitude of perturbation.
        data_grad (Tensor): Gradient of the loss w.r.t the image.
    Returns:
        perturbed_image (Tensor): The adversarial example.
    """
    # Get the sign of the gradients
    sign_data_grad = data_grad.sign()
    # Create the perturbed image by adjusting each pixel
    perturbed_image = image + epsilon * sign_data_grad
    # Compute the per-channel min and max in normalized space.
    clamp_min = torch.tensor([(0 - m) / s for m, s in zip(imagenet_mean, imagenet_std)],
                             device=image.device).view(1, 3, 1, 1)
    clamp_max = torch.tensor([(1 - m) / s for m, s in zip(imagenet_mean, imagenet_std)],
                             device=image.device).view(1, 3, 1, 1)
    # Clamp to ensure values remain in the valid range for normalized images.
    perturbed_image = torch.max(torch.min(perturbed_image, clamp_max), clamp_min)
    return perturbed_image

# -------------------------------
# 3. Testing with Adversarial Attack
# -------------------------------
def test(model, device, test_loader, epsilon):
    """
    Runs a simple adversarial attack on the test set using FGSM and reports the robust accuracy.
    Only samples that are initially correctly classified are attacked.
    Args:
        model: The pretrained neural network.
        device: Device to run the model.
        test_loader: DataLoader for the test set.
        epsilon: Perturbation magnitude.
    Returns:
        final_acc: Accuracy of the model on adversarial examples (as a fraction).
        adv_examples: A list of examples for visualization.
    """
    correct = 0
    adv_total = 0
    adv_examples = []
    # Loop over all examples in the test set
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        data.requires_grad = True
        # Forward pass
        output = model(data)
        init_pred = output.max(1, keepdim=True)[1]
        # Only attack if the initial prediction is correct
        if init_pred.item() != target.item():
            continue
        adv_total += 1
        # Calculate loss and backpropagate
        loss = criterion(output, target)
        model.zero_grad()
        loss.backward()
        data_grad = data.grad.data
        # Generate adversarial example
        perturbed_data = fgsm_attack(data, epsilon, data_grad)
        # Re-classify the perturbed image
        output = model(perturbed_data)
        final_pred = output.max(1, keepdim=True)[1]
        # Check if attack succeeded (i.e. model got it wrong) or not
        if final_pred.item() == target.item():
            correct += 1
            # Save some examples for visualization (only a few examples are stored)
            if epsilon == 0 and len(adv_examples) < 5:
                # Unnormalize for visualization
                adv_ex = unnormalize(perturbed_data.squeeze().detach().cpu())
                adv_examples.append((init_pred.item(), final_pred.item(), adv_ex.numpy()))
        else:
            if len(adv_examples) < 5:
                adv_ex = unnormalize(perturbed_data.squeeze().detach().cpu())
                adv_examples.append((init_pred.item(), final_pred.item(), adv_ex.numpy()))
    final_acc = correct / float(adv_total) if adv_total > 0 else 0
    print("Epsilon: {}\tTest Accuracy = {} / {} = {:.2f}%".format(epsilon, correct, adv_total, final_acc*100))
    return final_acc, adv_examples

# -------------------------------
# 4. Run the Attack and Visualize Examples
# -------------------------------
# Set the perturbation magnitude
epsilon = 0.1
acc, examples = test(model, device, testloader, epsilon)
# Plot some adversarial examples
plt.figure(figsize=(3, 3))
for i, (orig, adv, ex_img) in enumerate(examples):
    plt.subplot(1, len(examples), i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.title("{} -> {}".format(orig, adv))
    # Convert from (C, H, W) to (H, W, C) for display
    plt.imshow(np.transpose(ex_img, (1, 2, 0)))
plt.tight_layout()
plt.show()


### **1.5. Taxonomies and Models for Attack Classification**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20A/image_01_05.png" width="250">



>* Taxonomies categorize and analyze attack vectors.
* STRIDE model identifies AI vulnerabilities effectively.

>* MITRE ATT&CK maps attack lifecycle stages.
* Enhances AI resilience against adversarial threats.

>* Emerging taxonomies address AI-specific threats.
* ATLAS categorizes attacks by impact on AI.



## **2. Exploring Weaknesses in AI Models and Data**

### **2.1. Identifying Flaws in Data Preparation Processes**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20A/image_02_01.png" width="250">



>* Biased data affects model accuracy and fairness.
* Ensure datasets are comprehensive and inclusive.

>* Missing data skews AI model results.
* Data imputation requires careful application.

>* Inadequate data cleaning affects model accuracy.
* Preprocessing ensures data quality and consistency.



### **2.2. Balancing Model Complexity and Generalization**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20A/image_02_02.png" width="250">



>* Complex models risk overfitting; simple models underfit.
* Use techniques to balance complexity and generalization.

>* Complex models capture noise, not trends.
* Simpler models outperform by focusing on indicators.

>* Dynamic data challenges model adaptability.
* Continuous learning ensures resilience and security.



### **2.3. Mitigating Insider Risks and Strengthening Access Controls**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20A/image_02_03.png" width="250">



>* Insider threats risk AI data integrity.
* Implement RBAC and MFA for security.

>* Promote security awareness through regular training.
* Monitor user activities for suspicious behavior.

>* Collaborate with cybersecurity firms for threat intelligence.
* Share threat intelligence with industry peers.



### **2.4. Case Studies of Past Breaches and Insights**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20A/image_02_04.png" width="250">



>* AI systems vulnerable to adversarial attacks.
* Need robust datasets and model architectures.

>* Tay chatbot manipulated via social engineering tactics.
* Highlights need for content moderation and monitoring.

>* Clearview AI breach exposes data vulnerabilities.
* Highlights need for ethical AI security measures.



### **2.5. Real-Time Threat Detection and Response**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20A/image_02_05.png" width="250">



>* Continuous monitoring for threat identification.
* Automated responses to detected threats.

>* AI enhances threat prediction and identification.
* Continuous learning improves security against threats.

>* Scalable solutions for large data monitoring.
* Collaboration enhances threat detection effectiveness.



## **3. Strengthening AI Systems: Architecture, Surveillance, and Countermeasures**

### **3.1. Implementing Robust Authentication and Authorization Protocols**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20A/image_03_01.png" width="250">



>* Implement multi-factor authentication for user security.
* Use role-based access control for data protection.

>* Tailor protocols for AI model challenges.
* Use dynamic authorization for adaptive security.

>* AI systems require secure authentication methods.
* Protect sensitive data in finance, healthcare, government.



### **3.2. AI-Specific Intrusion Detection Techniques**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20A/image_03_02.png" width="250">



>* AI techniques detect threats in AI systems.
* Unsupervised learning adapts to new threats.

>* Integrate AI IDS with cybersecurity tools.
* Protect patient data with AI-driven IDS.

>* Continuous learning adapts to evolving threats.
* Secures critical infrastructure in smart cities.



### **3.3. Balancing Automation and Human Oversight in Incident Response**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20A/image_03_03.png" width="250">



>* Automation enhances speed and efficiency in response.
* Human oversight crucial for complex threat interpretation.

>* Automated systems flag unusual transaction patterns.
* Human analysts assess and respond to threats.

>* Invest in workforce training for AI management.
* Foster continuous learning in AI advancements.



### **3.4. Integrating Cybersecurity Expertise in AI Development**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20A/image_03_04.png" width="250">



>* Collaborate early with cybersecurity professionals.
* Embed security in AI design phase.

>* Regular audits and penetration testing conducted.
* Continuous collaboration enhances AI system security.

>* Foster security awareness through training programs.
* Cultivate a security-first mindset among stakeholders.



### **3.5. Continuous Security Enhancement Practices**

<img src="https://storage.googleapis.com/mrafiei-odu/module_07/Lecture%20A/image_03_05.png" width="250">



>* Regular updates protect AI from vulnerabilities.
* Structured patching ensures operational continuity.

>* Adopt proactive threat intelligence frameworks.
* Integrate threat intelligence into AI strategies.

>* Regular training on latest security practices.
* Cultivate workforce for innovative security solutions.



# <font color="#418FDE" size="6.5" uppercase>**Adversarial Attacks and Security in AI**</font>


In this lecture, you learned to:
- Describe different types of adversarial attacks on AI models.
- Assess vulnerabilities in models and datasets.
- Propose security strategies to mitigate these threats.

In the next Lecture (Lecture B), we will go over 'Privacy and Privacy-Enhancing Technologies'